In [1]:
import pandas as pd
import plotly.express as px

In [ ]:
data = pd.read_csv('data/raw/dataset.csv')[['statement', 'status']].rename(columns={'status': 'target'})

### Data Cleaning

First check all missing values, missing statements are observed all classes.

These observations will not provide any meaningful insight for the model (i.e. same input, but different output - it will negative affect the learning of input-output mapping), thus we drop these.

In [3]:
data[data.statement.isna()].groupby(by='target').size()

target
Anxiety                  47
Bipolar                 100
Normal                    8
Personality disorder    124
Stress                   82
Suicidal                  1
dtype: int64

In [4]:
data = data[~data.statement.isna()].reset_index(drop=True)
data

,statement,target
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...
52676,Nobody takes me seriously I’ve (24M) dealt wit...,Anxiety
52677,"selfishness ""I don't feel very good, it's lik...",Anxiety
52678,Is there any way to sleep better? I can't slee...,Anxiety
52679,"Public speaking tips? Hi, all. I have to give ...",Anxiety


Next, we check for entirely duplicated data (same data, same target).

We do not need duplicated data for processing now, thus drop them.

In [5]:
data.drop_duplicates(inplace=True, ignore_index=True) # inplace parameter True means the method does not create a copy of the dataframe but performs the operation in-memory
data

,statement,target
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...
51088,Anxiety cause faintness when standing up ? As ...,Anxiety
51089,anxiety heart symptom does anyone else have th...,Anxiety
51090,Travel Anxiety Hi all! Long time anxiety suffe...,Anxiety
51091,fomo from things i’m not involved in does anyo...,Anxiety


Next, we check for data with the same inputs but different target labels.

Including these for processing and modelling would negative affect the learning of input-output mapping, drop as well.

In this case, unlike earlier, we do not just drop the duplicated instance but both instances because they have the same statement but different label.

Doing so would prevent introducing our own biases into the dataset (i.e. what we think a statement should be labelled).

In [6]:
# number of unique statements does not equal the number of data samples
data.statement.nunique() == data.shape[0]

False

While we acknowledge that a person could be assigned multiple mental health labels, the number of multi-label samples are sparse compared to single-label samples.

This could affect the model performance due to insufficient examples and increase the probability of mislabels, hence confine the modelling to single-label and remove these.

In [7]:
data[data.statement.duplicated(keep=False)]

,statement,target
8042,"All this work, all this pressure that everyone...",Suicidal
8043,"All this work, all this pressure that everyone...",Depression
8936,#NAME?,Depression
10348,Recently I have started this internship and mo...,Suicidal
10353,Recently I have started this internship and mo...,Depression
11875,I am a drug abuser (benzos and meth mostly) an...,Suicidal
11879,I am a drug abuser (benzos and meth mostly) an...,Depression
12210,So...HI I am Emma a young trans woman from ger...,Suicidal
12214,So...HI I am Emma a young trans woman from ger...,Depression
12307,I have been at home the past year... quite lit...,Suicidal


In [8]:
data = data[~data.statement.duplicated(keep=False)].reset_index(drop=True)
data

,statement,target
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...
51050,Anxiety cause faintness when standing up ? As ...,Anxiety
51051,anxiety heart symptom does anyone else have th...,Anxiety
51052,Travel Anxiety Hi all! Long time anxiety suffe...,Anxiety
51053,fomo from things i’m not involved in does anyo...,Anxiety


In [9]:
counts = data.groupby(by='target').count().reset_index()
counts

,target,statement
0,Anxiety,3617
1,Bipolar,2501
2,Depression,15078
3,Normal,16037
4,Personality disorder,895
5,Stress,2293
6,Suicidal,10634


In [23]:
fig = px.bar(
    counts,
    x='statement',
    y='target',
    orientation='h',
    title='Frequency of Mental Health Labels',
    width=1000,
    height=600
)
fig.update_layout(title_x=0.5, yaxis_title='Label', xaxis_title='Number of Samples', title_font=dict(family='Arial', size=25))
fig.show()

In [27]:
data.to_csv('data/clean/cleaned_data.csv', index=False)